# PL7 — Context Engineering & RAG

### Generative AI · MSc in Artificial Intelligence · Lab 7

You will build, measure and improve a small RAG assistant for the fictional *Universidade de Vale Alto (UVA)*. The corpus is in **European Portuguese**, which makes model choice (multilingual embeddings, rerankers) part of the exercise.

**Learning goals**
1. Measure how a context window is spent and shrink it with *select* and *compress*.
2. Observe position effects ("lost in the middle") and write a structured, citation-friendly prompt.
3. Compare chunking strategies.
4. Build and evaluate dense, sparse (BM25), hybrid (RRF) and reranked retrieval with Recall@k and MRR.
5. Generate grounded answers with citations, metadata filtering and abstention, and evaluate them.

| Part | Topic | Time |
|---|---|---|
| 0 | Setup | 10 min |
| 1 | Context engineering | 25 min |
| 2 | Chunking | 10 min |
| 3 | Dense retrieval & metrics | 20 min |
| 4 | BM25 & hybrid search | 15 min |
| 5 | Reranking | 10 min |
| 6 | Grounded generation & evaluation | 20 min |
| 7 | Results & report | 10 min |
| ★ | Bonus: query rewriting, HyDE, agentic loop | at home |

**Deliverable:** this notebook, executed, with the results table in Part 7 and short answers to every ❓ question.

**Instructions.** Run the cells in order. Cells marked `# TODO` are yours to complete.

**Names / student numbers:** _…_

## Part 0 · Setup (10 min)

Use **Google Colab with a GPU runtime** (*Runtime → Change runtime type → T4 GPU*). Two LLM backends are supported:

* `"hf"`: a small open model running locally (free). Use `Qwen/Qwen2.5-1.5B-Instruct` on GPU, `Qwen/Qwen2.5-0.5B-Instruct` if you only have a CPU.
* `"anthropic"`: the Claude API, if you have a key. Faster and much stronger, which changes some results. Note which one you used in your report.

In [ ]:
!pip -q install sentence-transformers rank_bm25 transformers accelerate anthropic

In [ ]:
import os, re, json, time, math, random, unicodedata
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(0); np.random.seed(0)

In [ ]:
BACKEND = "hf"                              # "hf" or "anthropic"
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"     # "Qwen/Qwen2.5-0.5B-Instruct" on CPU
ANTHROPIC_MODEL = "claude-haiku-5-5"        # check docs.claude.com for current model ids

if BACKEND == "hf":
    from transformers import pipeline
    _gen = pipeline("text-generation", model=HF_MODEL, dtype="auto", device_map="auto")
else:
    import anthropic, getpass
    if "ANTHROPIC_API_KEY" not in os.environ:
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
    _client = anthropic.Anthropic()

def llm(prompt, system="You are a helpful assistant.", max_tokens=300):
    """Single-turn call to the chosen backend. Deterministic (greedy) for reproducibility."""
    if BACKEND == "hf":
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
        out = _gen(msgs, max_new_tokens=max_tokens, do_sample=False)
        return out[0]["generated_text"][-1]["content"].strip()
    r = _client.messages.create(model=ANTHROPIC_MODEL, max_tokens=max_tokens, system=system,
                                messages=[{"role": "user", "content": prompt}])
    return r.content[0].text.strip()

print(llm("Diz olá numa frase curta."))

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")   # used only to count tokens

def n_tokens(text: str) -> int:
    return len(tok.encode(text))

def norm(text: str) -> str:
    """Lowercase and strip accents, for robust string matching."""
    text = unicodedata.normalize("NFKD", str(text).lower())
    return "".join(ch for ch in text if not unicodedata.combining(ch))

print(n_tokens("A propina anual é de 710 euros."), norm("Época Especial"))

### The corpus

`DOCS` holds 18 short UVA documents. Each has an `id`, `title`, `date` and `topic`. Two topics have **several versions** (`propinas`, `reembolso`), which you will need later. `LONG_REG` is a longer regulation you will chunk yourself in Part 2. `GOLD` is a hand-labelled evaluation set: question, ids of the relevant chunks, and a key string the answer must contain.

In [ ]:
def D(id, title, date, text, topic=None):
    return {"id": id, "title": title, "date": date, "topic": topic or id, "text": text}

DOCS = [
    D("propinas-2024", "Propinas 2024/2025", "2024-07-01",
      "A propina anual para licenciaturas e mestrados integrados é de 697 euros, paga em 10 prestações mensais de outubro a julho. O não pagamento de três prestações consecutivas suspende a inscrição.", "propinas"),
    D("propinas-2026", "Propinas 2026/2027", "2026-07-01",
      "A propina anual para licenciaturas e mestrados integrados é de 710 euros. Pode ser paga numa prestação única, com 2% de desconto, ou em 10 prestações mensais. Um atraso superior a 60 dias implica a suspensão da inscrição.", "propinas"),
    D("reembolso-2023", "Política de reembolsos (versão 2023)", "2023-09-01",
      "Os estudantes que anulem a inscrição têm direito ao reembolso das prestações de propina já pagas se o pedido for feito até 14 dias após o início do semestre.", "reembolso"),
    D("reembolso-2026", "Política de reembolsos (versão 2026)", "2026-09-15",
      "O prazo para pedir o reembolso das prestações de propina após anulação da inscrição passa a ser de 30 dias após o início do semestre. O pedido é feito no portal académico, na secção Requerimentos, e o reembolso é processado em até 45 dias úteis.", "reembolso"),
    D("uc-4471", "Ficha da UC-4471 Aprendizagem Automática", "2026-07-15",
      "Unidade curricular UC-4471 Aprendizagem Automática: 6 ECTS, 2.º ano, 1.º semestre. Docente responsável: Prof.ª Helena Matos. Avaliação: projeto (50%) e exame (50%), com nota mínima de 8 valores em cada componente."),
    D("uc-4472", "Ficha da UC-4472 Processamento de Linguagem Natural", "2026-07-15",
      "Unidade curricular UC-4472 Processamento de Linguagem Natural: 6 ECTS, 3.º ano, 2.º semestre. Docente responsável: Prof. Rui Carvalho. Avaliação: dois trabalhos práticos (60%) e um teste escrito (40%). Pré-requisito recomendado: UC-4471."),
    D("clu-77", "Acesso ao cluster de GPUs (serviço CLU-77)", "2025-10-01",
      "Estudantes de mestrado e doutoramento podem pedir acesso ao cluster de GPUs através do formulário CLU-77 no portal de serviços informáticos. Cada projeto tem uma quota de 200 horas de GPU por semestre. Pedidos de estudantes de licenciatura exigem uma declaração do orientador."),
    D("ia-trabalhos", "Uso de IA generativa em trabalhos académicos", "2025-09-01",
      "É permitido usar ferramentas de IA generativa para apoio à escrita e à programação, desde que o uso seja declarado numa secção própria do trabalho. A submissão de texto gerado por IA como próprio, sem declaração, é considerada fraude académica (ver Regulamento Pedagógico, artigo 9.º)."),
    D("biblioteca", "Horários da Biblioteca Central", "2026-09-01",
      "A Biblioteca Central está aberta de segunda a sexta das 8h30 às 23h00 e aos sábados das 9h00 às 18h00. Em época de exames abre também ao domingo. O empréstimo domiciliário é de 15 dias, renovável duas vezes."),
    D("eduroam", "Rede sem fios eduroam", "2025-02-01",
      "A universidade disponibiliza a rede eduroam em todos os edifícios. Configuração: utilizador no formato numero@alunos.uva.pt e a palavra-passe do portal académico. Problemas devem ser reportados ao helpdesk pela extensão 5050."),
    D("erasmus", "Programa Erasmus+", "2026-01-05",
      "As candidaturas para mobilidade Erasmus+ no ano letivo seguinte decorrem de 1 a 31 de janeiro. Requisitos: ter concluído pelo menos 60 ECTS e média mínima de 12 valores. A bolsa mensal varia entre 300 e 500 euros consoante o país de destino."),
    D("trabalhador-estudante", "Estatuto de trabalhador-estudante", "2025-09-01",
      "O estatuto de trabalhador-estudante é pedido no início de cada semestre, com contrato de trabalho ou declaração da entidade empregadora. Dá direito a época especial de exames, dispensa de frequência mínima e escolha preferencial de horários."),
    D("cantina", "Cantina e refeições sociais", "2026-09-01",
      "A cantina serve almoço das 12h00 às 14h30 e jantar das 19h00 às 21h00. A refeição social custa 3,10 euros. Existe sempre uma opção vegetariana."),
    D("alojamento", "Residências universitárias", "2026-06-01",
      "As candidaturas a alojamento nas residências são feitas em julho para o ano letivo seguinte. Os bolseiros têm prioridade. O preço mensal de um quarto duplo é de 85 euros para bolseiros e 160 euros para não bolseiros."),
    D("bolsas", "Bolsas de estudo de ação social", "2026-05-20",
      "A candidatura à bolsa de estudo é feita online na plataforma da DGES entre junho e setembro. O valor da bolsa depende do rendimento per capita do agregado familiar e cobre pelo menos o valor da propina."),
    D("calendario-2026", "Calendário escolar 2026/2027", "2026-06-30",
      "O 1.º semestre de 2026/2027 começa a 14 de setembro de 2026 e termina a 22 de janeiro de 2027. A época normal de exames do 1.º semestre decorre de 4 a 22 de janeiro. O 2.º semestre começa a 15 de fevereiro de 2027."),
    D("apoio-psicologico", "Gabinete de apoio psicológico", "2025-09-01",
      "O gabinete de apoio psicológico oferece consultas gratuitas e confidenciais, marcadas por e-mail para gap@uva.pt. Em situações urgentes existe atendimento sem marcação às quartas-feiras das 14h00 às 17h00."),
    D("portal", "Portal académico", "2025-09-01",
      "No portal académico é possível consultar notas, fazer inscrições em exames, submeter requerimentos e descarregar declarações de matrícula. A inscrição em exames da época de recurso deve ser feita até 5 dias úteis antes da data do exame."),
]

LONG_REG = """Regulamento Pedagógico da Universidade de Vale Alto (versão de setembro de 2025)

Artigo 1.º – Objeto
O presente regulamento define as regras de avaliação aplicáveis aos cursos de 1.º e 2.º ciclo e aos mestrados integrados da Universidade de Vale Alto. Aplica-se a todas as unidades curriculares, salvo disposição em contrário na respetiva ficha.

Artigo 2.º – Escala de classificação
A classificação das unidades curriculares é expressa numa escala inteira de 0 a 20 valores. Considera-se aprovado o estudante que obtenha classificação final igual ou superior a 10 valores. As classificações intermédias são arredondadas às décimas.

Artigo 3.º – Avaliação contínua
A avaliação contínua inclui pelo menos dois elementos de avaliação distribuídos ao longo do semestre. Nas unidades curriculares com componente prática é exigida a presença em pelo menos 75% das aulas práticas, exceto para estudantes com estatuto especial.

Artigo 4.º – Épocas de exame
Existem três épocas de exame: época normal, época de recurso e época especial. O estudante reprovado ou que não compareça na época normal pode apresentar-se à época de recurso, mediante inscrição prévia no portal académico.

Artigo 5.º – Época especial
Têm acesso à época especial os trabalhadores-estudantes, os atletas de alta competição, os dirigentes associativos e os finalistas a quem faltem no máximo duas unidades curriculares, até ao limite de 12 ECTS, para concluir o curso.

Artigo 6.º – Melhoria de classificação
O estudante pode requerer melhoria de classificação uma única vez em cada unidade curricular, na época de recurso do ano letivo seguinte ao da aprovação. Prevalece a classificação mais elevada. A melhoria está sujeita ao pagamento de uma taxa fixada anualmente.

Artigo 7.º – Revisão de prova
O estudante pode requerer a revisão de uma prova escrita no prazo de 3 dias úteis após a publicação da classificação. O pedido está sujeito a uma taxa de 25 euros, reembolsada se a classificação for alterada a favor do estudante.

Artigo 8.º – Prova oral complementar
Quando a classificação de um exame escrito se situe entre 8 e 9,4 valores, o docente responsável pode convocar o estudante para uma prova oral complementar, que decide a aprovação.

Artigo 9.º – Fraude académica
Constituem fraude académica o plágio, a cópia, a utilização de meios não autorizados durante as provas e a apresentação como próprio de conteúdo gerado por ferramentas de inteligência artificial sem a devida declaração. A fraude implica a anulação da prova e a abertura de processo disciplinar. A reincidência pode conduzir à suspensão até dois semestres.

Artigo 10.º – Publicação de classificações
As classificações são publicadas no portal académico no prazo máximo de 15 dias úteis após a realização da prova.

Artigo 11.º – Casos omissos
Os casos omissos são decididos pelo Conselho Pedagógico, ouvido o docente responsável pela unidade curricular.
"""

def G(q, relevant, answer): return {"q": q, "relevant": relevant, "answer": answer}

GOLD = [
    G("Qual é o prazo para pedir o reembolso das propinas?", ["reembolso-2026"], "30 dias"),
    G("Quanto custa a propina anual em 2026/2027?", ["propinas-2026"], "710"),
    G("Quem é a docente responsável pela UC-4471?", ["uc-4471"], "helena matos"),
    G("Quantas horas de GPU tem cada projeto no CLU-77?", ["clu-77"], "200"),
    G("Posso usar o ChatGPT para me ajudar a escrever o relatório?", ["ia-trabalhos"], "declar"),
    G("O que acontece se eu entregar um trabalho feito por IA sem avisar?", ["ia-trabalhos", "regped-art-9"], "fraude"),
    G("Quantas vezes posso tentar subir a nota de uma cadeira?", ["regped-art-6"], "uma unica"),
    G("Qual é a nota mínima para passar a uma cadeira?", ["regped-art-2"], "10"),
    G("Quem tem direito a época especial de exames?", ["regped-art-5", "trabalhador-estudante"], "trabalhador"),
    G("Até quando posso pedir para rever o meu exame?", ["regped-art-7"], "3 dias"),
    G("A que horas fecha a biblioteca ao sábado?", ["biblioteca"], "18"),
    G("Como me ligo à internet sem fios no campus?", ["eduroam"], "eduroam"),
    G("Que média preciso para fazer Erasmus?", ["erasmus"], "12"),
    G("Quanto paga um bolseiro por um quarto na residência?", ["alojamento"], "85"),
    G("Quando começa o primeiro semestre de 2026/2027?", ["calendario-2026"], "14 de setembro"),
    G("Qual é o peso dos trabalhos práticos na avaliação de PLN?", ["uc-4472"], "60"),
    G("Qual é o número de telemóvel do reitor?", [], "nao sei"),   # unanswerable: must abstain
]
print(len(DOCS), "docs |", n_tokens(LONG_REG), "tokens in LONG_REG |", len(GOLD), "gold questions")

## Part 1 · Context engineering (25 min)

### 1.1 Where do the tokens go? (8 min)

Below is the context a naive support agent sends on **every** turn: a system prompt, 32 tool definitions, the last 40 messages of conversation (20 exchanges), every document in the knowledge base and, finally, the user's question.

In [ ]:
SYSTEM = ("És o assistente académico da Universidade de Vale Alto. Responde de forma clara, educada e concisa, "
          "em português europeu. Segue as regras institucionais, não inventes informação e encaminha casos "
          "sensíveis para os serviços competentes. ") * 4

TOOL_NAMES = ["get_grades", "update_address", "book_room", "cancel_booking", "list_courses", "enrol_exam",
    "unenrol_exam", "get_timetable", "request_certificate", "pay_fee", "get_fees", "search_regulations",
    "open_ticket", "close_ticket", "get_menu", "reserve_meal", "library_search", "renew_loan", "get_calendar",
    "erasmus_status", "scholarship_status", "upload_document", "change_password", "get_wifi_config",
    "contact_tutor", "book_counselling", "list_events", "parking_permit", "lost_and_found", "print_quota",
    "gpu_quota", "send_email"]
TOOLS = [{"name": n, "description": f"Tool that lets the assistant {n.replace('_', ' ')} in the UVA systems for the current student.",
          "input_schema": {"type": "object", "properties": {"student_id": {"type": "string"}, "details": {"type": "string"}},
                           "required": ["student_id"]}} for n in TOOL_NAMES]
TOOLS_TXT = json.dumps(TOOLS, ensure_ascii=False, indent=1)

HISTORY = []   # filler only: the assistant turns are not meant to answer the questions
for g in GOLD[4:16] * 2:
    HISTORY += [f"Utilizador: {g['q']}", "Assistente: Segundo os documentos da universidade, " + DOCS[len(HISTORY) % len(DOCS)]["text"]]
HISTORY = HISTORY[-40:]
HISTORY_TXT = "\n".join(HISTORY)

DOCS_TXT = "\n\n".join(f"[{d['id']}] {d['title']}\n{d['text']}" for d in DOCS)
QUERY = "Qual é o prazo para pedir o reembolso das propinas?"

COMPONENTS = {"system": SYSTEM, "tools": TOOLS_TXT, "history": HISTORY_TXT, "documents": DOCS_TXT, "query": QUERY}

In [ ]:
def budget_table(components: dict) -> pd.DataFrame:
    rows = [{"component": k, "tokens": n_tokens(v)} for k, v in components.items()]
    df = pd.DataFrame(rows)
    df["share_%"] = (100 * df.tokens / df.tokens.sum()).round(1)
    return df

df = budget_table(COMPONENTS)
display(df)
print("TOTAL:", df.tokens.sum())
df.set_index("component").tokens.plot.barh(title="Tokens per component"); plt.show()

❓ **Q1.** What share of the window is the user's actual question? Which component would you attack first, and with which strategy (*write, select, compress, isolate*)?

Now apply **select** and **compress** to bring the total **under 1,500 tokens** while keeping what the query needs. Suggested moves: keep only tools relevant to the query, keep the last 4 history lines, keep only the 3 documents with the largest word overlap with the query. Do not remove the query.

**Resposta Q1:** A pergunta do utilizador é uma fração ínfima da janela, bem abaixo de 1% (17 tokens num total de vários milhares). O componente que atacaria primeiro são as ferramentas, que ocupam quase metade da janela: das 32 definições, só 3 ou 4 servem para uma pergunta sobre propinas e reembolsos. A estratégia é **select**: enviar só as ferramentas relevantes. A seguir viriam o histórico (**compress**, ficando só com os últimos turnos ou um resumo) e os documentos (**select**, via RAG).

In [ ]:
def overlap(a: str, b: str) -> int:
    wa = set(re.findall(r"\w+", norm(a))); wb = set(re.findall(r"\w+", norm(b)))
    return len(wa & wb)

def shrink(components: dict, query: str) -> dict:
    system = SYSTEM[:len(SYSTEM) // 4]
    keep_tools = {"get_fees", "pay_fee", "search_regulations", "open_ticket"}
    tools = json.dumps([t for t in TOOLS if t["name"] in keep_tools], ensure_ascii=False)
    history = "\n".join(HISTORY[-4:])
    top = sorted(DOCS, key=lambda d: overlap(query, d["title"] + " " + d["text"]), reverse=True)[:3]
    documents = "\n\n".join(f"[{d['id']}] {d['title']}\n{d['text']}" for d in top)
    return {"system": system, "tools": tools, "history": history, "documents": documents, "query": query}

small = shrink(COMPONENTS, QUERY)
df2 = budget_table(small); display(df2); print("TOTAL:", df2.tokens.sum())
print("Docs kept:", re.findall(r"^\[(.+?)\]", small["documents"], flags=re.M))

❓ **Q2.** Which three documents did the word-overlap rule keep? Is `reembolso-2026` among them? Is there an **outdated version** of a document, or a document that has nothing to do with the question? Print `overlap()` for each kept document: which words caused the match? Which failure modes from the lecture does this illustrate, and what would make the selection better?

### 1.2 Lost in the middle (9 min)

We hide one relevant document (the "needle") among the other 17 and move it from the start to the end of the context. For each position we ask 5 questions and count correct answers.

In [ ]:
for line in small["documents"].split("\n\n"):
    did = re.match(r"\[(.+?)\]", line).group(1)
    d = ID2DOC[did]
    wa = set(re.findall(r"\w+", norm(QUERY))); wb = set(re.findall(r"\w+", norm(d["title"] + " " + d["text"])))
    print(f"{did:20s} overlap={overlap(QUERY, d['title'] + ' ' + d['text'])}  palavras: {sorted(wa & wb)}")

**Resposta Q2:** A regra manteve `reembolso-2026`, `propinas-2024` e `apoio-psicologico`. O `reembolso-2026` está entre eles e é o que tem mais palavras em comum (prazo, pedir, reembolso…). Mas o `propinas-2024` é uma **versão desatualizada** (existe a de 2026) e nem sequer fala de reembolsos: entra só por causa de "propinas" e de palavras como "o", "e", "para". O `apoio-psicologico` não tem nada a ver com a pergunta e entra apenas por stopwords ("das", "e", "o", "para").

Isto ilustra os modos de falha **distraction/confusion** (conteúdo irrelevante no contexto) e o risco de **clash** (versões antigas misturadas com as novas). Melhoraria a seleção removendo stopwords, usando BM25 ou embeddings em vez de contar palavras, e aplicando um filtro de metadados que fique só com a versão mais recente de cada tópico.

In [ ]:
NEEDLE_TESTS = [  # (question, needle doc id, key string in a correct answer)
    ("Quem é a docente responsável pela UC-4471?", "uc-4471", "helena matos"),
    ("Quantas horas de GPU tem cada projeto no CLU-77?", "clu-77", "200"),
    ("A que horas fecha a biblioteca ao sábado?", "biblioteca", "18"),
    ("Que média preciso para fazer Erasmus?", "erasmus", "12"),
    ("Quanto paga um bolseiro por um quarto na residência?", "alojamento", "85"),
]
ID2DOC = {d["id"]: d for d in DOCS}

def format_docs(docs):
    return "\n\n".join(f"[{d['id']}] {d['title']}\n{d['text']}" for d in docs)

In [ ]:
def build_context(needle_id: str, position: float) -> list:
    others = [d for d in DOCS if d["id"] != needle_id]
    random.Random(42).shuffle(others)
    others.insert(round(position * len(others)), ID2DOC[needle_id])
    return others

assert build_context("uc-4471", 0.0)[0]["id"] == "uc-4471"
assert build_context("uc-4471", 1.0)[-1]["id"] == "uc-4471"

In [ ]:
POSITIONS = [0.0, 0.25, 0.5, 0.75, 1.0]
acc = []
for p in POSITIONS:
    ok = 0
    for q, needle, key in NEEDLE_TESTS:
        ctx = format_docs(build_context(needle, p))
        ans = llm(f"Documentos:\n{ctx}\n\nPergunta: {q}\nResponde numa frase curta.", max_tokens=60)
        ok += key in norm(ans)
    acc.append(ok / len(NEEDLE_TESTS))
    print(f"position {p:.2f}: {ok}/{len(NEEDLE_TESTS)}")

plt.plot(POSITIONS, acc, marker="o"); plt.ylim(0, 1.05)
plt.xlabel("relative position of the needle"); plt.ylabel("accuracy"); plt.title("Lost in the middle?"); plt.show()

❓ **Q3.** Do you see a U-shape? Our context is only ~1.6k tokens and we use 5 questions, so results are noisy. What would you change to make the experiment more convincing (context length, number of questions, model size)? 

### 1.3 A structured, citation-friendly prompt (8 min)

Write `build_prompt(question, docs)` following the lecture: documents first, wrapped in tags **with id, title and date**; question last. `SYSTEM_RAG` asks for citations, abstention and the most recent version.

**Resposta Q3:** Não se vê um U claro: com um contexto de ~1,6 mil tokens o modelo encontra a agulha em quase todas as posições, e as diferenças que aparecem são de uma ou duas perguntas, ou seja, ruído. Para tornar a experiência convincente usaria contextos muito mais longos (dezenas de milhares de tokens, com muitos mais documentos distratores), muitas mais perguntas (dezenas ou centenas por posição) e mais posições intermédias, e compararia vários tamanhos de modelo. O efeito lost in the middle aparece sobretudo em contextos longos.

In [ ]:
SYSTEM_RAG = ("És o assistente académico da Universidade de Vale Alto. Responde em português europeu, numa ou duas frases, "
              "usando APENAS os documentos fornecidos. Cita o id de cada documento usado entre parênteses retos, por exemplo "
              "[id-do-documento]. Se existirem versões diferentes do mesmo documento, usa a mais recente (data mais tardia). "
              "Se a resposta não estiver nos documentos, responde exatamente: 'Não sei com base nos documentos disponíveis.'")

def build_prompt(question: str, docs: list) -> str:
    lines = ["<documents>"]
    for d in docs:
        lines += [f'<doc id="{d["id"]}" title="{d["title"]}" date="{d["date"]}">', d["text"], "</doc>"]
    lines += ["</documents>", "", "<question>", question, "</question>"]
    return "\n".join(lines)

def naive_prompt(question: str, docs: list) -> str:
    return "Contexto: " + " ".join(d["text"] for d in docs) + f"\nPergunta: {question}"

clash_docs = [ID2DOC["reembolso-2023"], ID2DOC["cantina"], ID2DOC["reembolso-2026"]]   # outdated version first
q = "Qual é o prazo para pedir o reembolso das propinas?"
print("NAIVE     :", llm(naive_prompt(q, clash_docs)))
print("STRUCTURED:", llm(build_prompt(q, clash_docs), system=SYSTEM_RAG))
print("ABSTAIN?  :", llm(build_prompt("Qual é o número de telemóvel do reitor?", clash_docs), system=SYSTEM_RAG))

❓ **Q4.** Compare the naive and the structured answers: which deadline does each give (14 or 30 days)? Did the structured prompt resolve the *clash* between the 2023 and 2026 policies? Did it cite in the requested `[id]` format? Did the model abstain on the unanswerable question? If it still fails, is that a prompt problem or something you should fix **before** the prompt (hint: Part 6)?

**Resposta Q4:** O prompt ingénuo junta os textos sem ids nem datas, por isso o modelo não tem como saber qual política é a mais recente e tende a responder 14 dias (a versão de 2023, que aparece primeiro). O prompt estruturado, com id, título e data em cada documento e a instrução de usar a versão mais recente, tem mais hipóteses de dar 30 dias e de citar `[reembolso-2026]`. Mas um modelo pequeno nem sempre resolve o clash nem cumpre o formato de citação, e pode não se abster na pergunta sem resposta.

Quando falha, não é sobretudo um problema de prompt: a versão desatualizada nem devia chegar ao modelo. O certo é resolver o conflito **antes** do prompt, com um filtro de metadados que fique só com a versão mais recente de cada tópico (`latest_only`, Parte 6).

## Part 2 · Chunking (10 min)

Implement two strategies for `LONG_REG` and compare them on structure (how many chunks mix articles or start mid-article) and on what they offer for retrieval and citations:

* `fixed_chunks`: windows of `size` words with `overlap` words of overlap.
* `article_chunks`: one chunk per *Artigo*, with a descriptive title (structure-aware chunking with a contextual header). Use the id format **`regped-art-{n}`**, which the gold set relies on.

In [ ]:
def fixed_chunks(text: str, size: int = 60, overlap: int = 15) -> list:
    words = text.split()
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return chunks

def article_chunks(text: str, doc_title="Regulamento Pedagógico", date="2025-09-01") -> list:
    out = []
    for piece in re.split(r"(?=Artigo \d+\.º)", text):
        m = re.match(r"Artigo (\d+)\.º\s*[–-]\s*(.+)", piece)
        if not m:
            continue
        n, heading = m.group(1), m.group(2).strip()
        cid = f"regped-art-{n}"
        out.append({"id": cid, "title": f"{doc_title}, Artigo {n}.º {heading}", "date": date,
                    "topic": cid, "text": piece.strip()})
    return out

In [ ]:
fixed = fixed_chunks(LONG_REG)
arts = article_chunks(LONG_REG)
mixed = sum(len(re.findall(r"Artigo \d+\.º", c)) >= 2 for c in fixed)
midstart = sum(not c.startswith("Artigo") for c in fixed[1:])   # the first chunk starts with the document title
print(f"fixed: {len(fixed)} chunks | {mixed} mix two articles | {midstart} of {len(fixed)-1} start mid-article")
print(f"article: {len(arts)} chunks ->", [c["id"] for c in arts][:4], "...")
print("\nExample fixed chunk:\n", fixed[5][:300])

CHUNKS = DOCS + arts
ID2CHUNK = {c["id"]: c for c in CHUNKS}
assert "regped-art-6" in ID2CHUNK, "article ids must look like regped-art-6"
print("\nIndex size:", len(CHUNKS), "chunks")

❓ **Q5.** Look at the example fixed-size chunk: it starts in the middle of an article heading (so the article number is lost) and runs into the next article. If it were retrieved alone, could the model tell which rule each sentence belongs to? How does the `title` field of `article_chunks` help, both for retrieval and for citations?

**Resposta Q5:** Não. Os excertos de tamanho fixo cortam o texto sem olhar para a estrutura: dos 10 excertos, 7 misturam dois artigos e 7 dos 9 seguintes começam a meio de um artigo. O exemplo começa em "de classificação O estudante pode requerer melhoria…", perdendo o "Artigo 6.º – Melhoria", e acaba já no "Artigo 7.º – Revisão". Sozinho, o modelo não saberia a que artigo pertence cada frase nem poderia citá-lo.

O `title` do `article_chunks` ("Regulamento Pedagógico, Artigo 6.º Melhoria de classificação") funciona como cabeçalho contextual. Na recuperação, junta ao texto palavras-chave do tema e do documento, o que ajuda tanto os embeddings como o BM25. Nas citações, cada excerto tem um id estável (`regped-art-6`) que corresponde a um artigo concreto e pode ser citado e verificado.

## Part 3 · Dense retrieval & metrics (20 min)

We use `intfloat/multilingual-e5-small`, a small multilingual bi-encoder. **E5 models expect prefixes:** `"query: "` for questions and `"passage: "` for documents.

In [ ]:
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer("intfloat/multilingual-e5-small")

def passage_text(c: dict) -> str:
    return f"{c['title']}. {c['text']}"

P_EMB = emb_model.encode(["passage: " + passage_text(c) for c in CHUNKS], normalize_embeddings=True)
print(P_EMB.shape)

In [ ]:
def dense_search(query: str, k: int = 10) -> list:
    q = emb_model.encode(["query: " + query], normalize_embeddings=True)[0]
    scores = P_EMB @ q
    return [CHUNKS[i]["id"] for i in np.argsort(-scores)[:k]]

print(dense_search("Quantas vezes posso tentar subir a nota de uma cadeira?", k=5))

Now the metrics. For a ranked list of ids and the set of relevant ids:

* **Recall@k** = |relevant ∩ top-k| / |relevant|
* **Reciprocal rank** = 1 / (rank of the first relevant id), or 0 if none is found. **MRR** is its mean over questions.

In [ ]:
def recall_at_k(ranked: list, relevant: list, k: int) -> float:
    if not relevant:
        return 0.0
    return len(set(relevant) & set(ranked[:k])) / len(relevant)

def reciprocal_rank(ranked: list, relevant: list) -> float:
    for pos, i in enumerate(ranked, start=1):
        if i in relevant:
            return 1.0 / pos
    return 0.0

assert recall_at_k(["a", "b", "c"], ["c", "z"], 2) == 0.0
assert recall_at_k(["a", "b", "c"], ["c", "z"], 3) == 0.5
assert reciprocal_rank(["a", "b", "c"], ["b"]) == 0.5

In [ ]:
RESULTS, MISSES = {}, {}

def evaluate(search_fn, name: str, depth: int = 10) -> dict:
    rows, t0 = [], time.time()
    for g in GOLD:
        if not g["relevant"]:
            continue                                    # unanswerable questions are not retrieval tests
        ranked = search_fn(g["q"], k=depth)
        rows.append({"q": g["q"], "R@1": recall_at_k(ranked, g["relevant"], 1),
                     "R@5": recall_at_k(ranked, g["relevant"], 5), "RR": reciprocal_rank(ranked, g["relevant"])})
    df = pd.DataFrame(rows)
    res = {"Recall@1": df["R@1"].mean(), "Recall@5": df["R@5"].mean(), "MRR": df["RR"].mean(),
           "ms/query": 1000 * (time.time() - t0) / len(df)}
    RESULTS[name] = {k: round(v, 3) for k, v in res.items()}
    MISSES[name] = df[df["R@5"] < 1]["q"].tolist()
    return RESULTS[name]

print(evaluate(dense_search, "dense"))
print("Not fully found in top-5:", MISSES["dense"])

❓ **Q6.** Which questions does dense retrieval miss? Look at their wording: exact codes (`UC-4471`, `CLU-77`), abbreviations (`PLN`), or paraphrases and colloquial words (*cadeira*, *subir a nota*)?

> **Read the metrics with care.** The evaluation has 16 answerable questions, so one question is worth 1/16 = 0.0625 of Recall@5. Differences of that size are one question, not a trend.

❓ **Q7 (2 min experiment).** Re-encode *without* the `query: ` / `passage: ` prefixes and re-evaluate as `"dense_noprefix"`. What happens? Is the difference larger than one question? Why does it matter to read the model card, even when a small test does not show the effect?

**Resposta Q6:** As falhas da pesquisa densa concentram-se nas perguntas com paráfrases e linguagem coloquial, que não usam as palavras do regulamento: "subir a nota de uma cadeira" (o artigo fala em "melhoria de classificação" e "unidade curricular"), "nota mínima para passar" ("aprovado", "10 valores") e "rever o meu exame" ("revisão de prova"). Códigos exatos como `UC-4471` e `CLU-77` e abreviaturas como `PLN` também são um ponto fraco dos embeddings, porque os tokens destes códigos têm pouco significado próprio.

**Resposta Q7:** Sem os prefixos `query:` / `passage:` os resultados pioram ou ficam iguais, e neste conjunto pequeno a diferença é de uma ou duas perguntas, por isso não é conclusiva. Mesmo assim, o modelo E5 foi treinado com esses prefixos e a model card diz para os usar: sem eles, as perguntas e os documentos ficam mal posicionados no espaço vetorial. Num corpus grande e realista o efeito aparece mesmo quando um teste com 16 perguntas não o mostra.

In [ ]:
P_EMB_NP = emb_model.encode([passage_text(c) for c in CHUNKS], normalize_embeddings=True)

def dense_noprefix(query: str, k: int = 10) -> list:
    q = emb_model.encode([query], normalize_embeddings=True)[0]
    scores = P_EMB_NP @ q
    return [CHUNKS[i]["id"] for i in np.argsort(-scores)[:k]]

print(evaluate(dense_noprefix, "dense_noprefix"))
print("dense         :", RESULTS["dense"])
print("noprefix misses:", MISSES["dense_noprefix"])

## Part 4 · BM25 & hybrid search (15 min)

BM25 needs a tokenizer. Ours lowercases, strips accents, keeps codes such as `uc-4471` intact and removes a few Portuguese stop words.

In [ ]:
from rank_bm25 import BM25Okapi

STOP = set(("a o os as de da do das dos e em um uma uns umas para por com que qual quais quem como no na nos nas "
            "ao aos se eu me meu minha posso pode quanto quanta quantos quantas onde quando sao ser ha").split())

def tokenize(text: str) -> list:
    return [t for t in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", norm(text)) if t not in STOP]

bm25 = BM25Okapi([tokenize(passage_text(c)) for c in CHUNKS])
print(tokenize("Quem é a docente responsável pela UC-4471?"))

In [ ]:
def bm25_search(query: str, k: int = 10) -> list:
    scores = bm25.get_scores(tokenize(query))
    return [CHUNKS[i]["id"] for i in np.argsort(-scores)[:k]]

def rrf(rankings: list, k: int = 60, top: int = 10) -> list:
    score = defaultdict(float)
    for ranking in rankings:
        for rank, d in enumerate(ranking, start=1):
            score[d] += 1.0 / (k + rank)
    return sorted(score, key=score.get, reverse=True)[:top]

def hybrid_search(query: str, k: int = 10, depth: int = 20) -> list:
    return rrf([dense_search(query, depth), bm25_search(query, depth)], top=k)

assert rrf([["a", "b"], ["b", "c"]])[0] == "b"
for name, fn in [("bm25", bm25_search), ("hybrid_rrf", hybrid_search)]:
    print(name, evaluate(fn, name))

In [ ]:
display(pd.DataFrame(RESULTS).T)
for name in ["dense", "bm25", "hybrid_rrf"]:
    print(f"{name:11s} misses:", MISSES[name])

In [ ]:
for kk in [10, 60, 100]:
    fn = lambda q, k=10, depth=20, kk=kk: rrf([dense_search(q, depth), bm25_search(q, depth)], k=kk, top=k)
    print(f"rrf k={kk:3d}:", evaluate(fn, f"hybrid_rrf_k{kk}"))
for kk in [10, 100]:
    RESULTS.pop(f"hybrid_rrf_k{kk}"); MISSES.pop(f"hybrid_rrf_k{kk}")
RESULTS.pop("hybrid_rrf_k60"); MISSES.pop("hybrid_rrf_k60")

❓ **Q8.** Which questions does BM25 win that dense loses, and vice versa? Does hybrid get the best of both? If not, why not on this corpus (look at which questions both methods miss, and at the corpus size)? Try `k=10` and `k=100` in `rrf`: does the constant matter here?

**Resposta Q8:** O BM25 ganha nas perguntas com termos exatos, como códigos (`UC-4471`, `CLU-77`) e palavras que aparecem tal e qual no documento ("biblioteca", "Erasmus", "residência"). Perde nas paráfrases: falha "subir a nota de uma cadeira", "nota mínima para passar a uma cadeira", "rever o meu exame" e o artigo 9.º na pergunta sobre trabalhos feitos por IA, porque o vocabulário da pergunta não coincide com o do regulamento. A pesquisa densa é o contrário: lida melhor com as paráfrases e pior com os códigos.

O híbrido junta o melhor dos dois quando um dos métodos coloca o documento certo bem no topo. Mas as perguntas que **ambos** falham continuam a falhar, porque o RRF só reordena o que os dois métodos encontram. Com um corpus de apenas 29 excertos, as listas de 20 candidatos contêm quase todo o corpus e a fusão muda pouco. Variar a constante do RRF (k = 10, 60, 100) praticamente não altera os resultados aqui: o k só muda o peso relativo das primeiras posições, e com tão poucos documentos a ordem final quase não muda.

## Part 5 · Reranking (10 min)

A multilingual **cross-encoder** reads (query, passage) *together* and outputs a relevance score. It is far too slow for the whole corpus, so it only reorders the top candidates from hybrid search.

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

In [ ]:
def rerank(query: str, ids: list, top: int = 5) -> list:
    scores = reranker.predict([(query, passage_text(ID2CHUNK[i])) for i in ids])
    order = np.argsort(-np.asarray(scores))
    return [ids[j] for j in order[:top]]

def hybrid_rerank(query: str, k: int = 10, depth: int = 15) -> list:
    return rerank(query, hybrid_search(query, k=depth), top=k)

print(evaluate(hybrid_rerank, "hybrid_rrf+rerank"))
display(pd.DataFrame(RESULTS).T)

In [ ]:
print("hybrid_rrf+rerank misses:", MISSES["hybrid_rrf+rerank"])

❓ **Q9.** How much did reranking improve Recall@1 and MRR, and what did it cost in ms/query? When would you skip it?

**Resposta Q9:** O reranking melhora sobretudo o Recall@1 e o MRR, porque o cross-encoder lê a pergunta e o excerto juntos e põe o documento certo em primeiro lugar em casos que o híbrido deixava em 2.º ou 3.º. O Recall@5 muda pouco, porque o reranking só reordena os candidatos que já lá estavam. O custo é o tempo: os ms/query sobem muito face ao híbrido, porque é preciso uma passagem do cross-encoder por cada par (pergunta, candidato).

Saltaria o reranking quando a latência é crítica, quando o corpus é muito pequeno e a primeira fase já acerta no topo, ou quando o LLM recebe vários excertos de qualquer forma e a ordem exata importa pouco.

## Part 6 · Grounded generation & evaluation (20 min)

### 6.1 Fix the clash before the prompt (5 min)

Write `latest_only(chunks)`: for each `topic`, keep only the chunk with the most recent `date`, preserving the original order. This is a **metadata filter**, cheaper and more reliable than asking the model to sort out versions.

In [ ]:
def latest_only(chunks: list) -> list:
    latest = {}
    for c in chunks:
        if c["topic"] not in latest or c["date"] > latest[c["topic"]]["date"]:
            latest[c["topic"]] = c
    keep = {id(c) for c in latest.values()}
    return [c for c in chunks if id(c) in keep]

print([c["id"] for c in latest_only([ID2CHUNK[i] for i in ["reembolso-2023", "cantina", "reembolso-2026"]])])
# expected: ['cantina', 'reembolso-2026']

### 6.2 The full pipeline (8 min)

`rag_answer` = hybrid retrieval → rerank → metadata filter → structured prompt → LLM → citation check.

In [ ]:
CITE_RE = re.compile(r"[\[(]([a-z0-9]+(?:-[a-z0-9]+)*)[\])]")      # [id] or (id)
STRICT_RE = re.compile(r"\[[a-z0-9]+(?:-[a-z0-9]+)*\]")             # the requested [id] format

ABSTAIN_PATTERNS = ["nao sei", "nao posso responder", "nao consigo responder", "nao tenho informacao",
                    "nao ha informacao", "nao existe informacao", "nao consta", "nao e possivel responder"]

def is_abstention(answer: str) -> bool:
    return any(p in norm(answer) for p in ABSTAIN_PATTERNS)

def rag_answer(question: str, k: int = 4) -> dict:
    ids = hybrid_rerank(question, k=8)
    chunks = latest_only([ID2CHUNK[i] for i in ids])[:k]
    answer = llm(build_prompt(question, chunks), system=SYSTEM_RAG, max_tokens=150)
    retrieved = [c["id"] for c in chunks]
    cited = sorted({c for c in CITE_RE.findall(answer) if c in ID2CHUNK or f"[{c}]" in answer})
    return {"question": question, "answer": answer, "retrieved": retrieved, "cited": cited,
            "strict_format": bool(STRICT_RE.search(answer)),
            "invalid_citations": sorted(c for c in cited if c not in retrieved),
            "abstained": is_abstention(answer)}

rag_answer("Qual é o prazo para pedir o reembolso das propinas?")

In [ ]:
def is_correct(gold: dict, a: dict) -> bool:
    if not gold["relevant"]:                       # unanswerable: correct = abstained
        return a["abstained"]
    return gold["answer"] in norm(a["answer"]) and not a["abstained"]

answers = [rag_answer(g["q"]) for g in GOLD]
ev = pd.DataFrame([{
    "question": a["question"][:55],
    "correct": is_correct(GOLD[i], a),
    "cites": len(a["cited"]) > 0,
    "[id] format": a["strict_format"],
    "invalid_cites": len(a["invalid_citations"]),
    "abstained": a["abstained"],
    "answer": a["answer"][:90],
} for i, a in enumerate(answers)])
display(ev)
print(f"Answer accuracy: {ev.correct.mean():.0%} | answers with citations: {ev.cites.mean():.0%} "
      f"(in the requested [id] format: {ev['[id] format'].mean():.0%}) | invalid citations: {ev.invalid_cites.sum()}")
print("Read the full answers too: the key string checks one fact, not the whole answer.")
for i, a in enumerate(answers):
    print(f"\n[{i}] {a['question']}\n    {a['answer']}")

### 6.3 LLM-as-a-judge for faithfulness (7 min)

String matching only checks the key fact. A judge checks whether **every** claim is supported by the retrieved context.

In [ ]:
JUDGE = """Avalia se a RESPOSTA é totalmente suportada pelos DOCUMENTOS. Não uses conhecimento externo.
Responde apenas com JSON, sem mais texto: {"faithful": true ou false, "unsupported": "frase não suportada ou vazio"}

DOCUMENTOS:
<<DOCS>>

RESPOSTA:
<<ANSWER>>"""

def judge(a: dict) -> dict:
    docs = format_docs([ID2CHUNK[i] for i in a["retrieved"]])
    raw = llm(JUDGE.replace("<<DOCS>>", docs).replace("<<ANSWER>>", a["answer"]), max_tokens=120)
    m = re.search(r"\{.*\}", raw, flags=re.S)
    try:
        return json.loads(m.group(0))
    except Exception:
        return {"faithful": None, "unsupported": "unparseable: " + raw[:80]}

ev["faithful"] = [None if a["abstained"] else judge(a).get("faithful") for a in answers]   # nothing to verify in an abstention
display(ev[["question", "correct", "faithful", "answer"]])
judged = ev[ev.faithful.notna()]
print(f"Judged faithful: {(judged.faithful == True).mean():.2f} of {len(judged)} non-abstaining answers")

❓ **Q10.** Find a case where `correct` and `faithful` disagree. Who is right, the string match or the judge? Read the answer yourself. What does this tell you about using a small model as a judge?

❓ **Q11.** Did the system abstain on the unanswerable question (the last one)? Did any answer cite a document that was not retrieved?

**Resposta Q10:** Há casos em que `correct` e `faithful` discordam. Por exemplo, uma resposta que contém a palavra-chave (`correct = True`) mas acrescenta um pormenor que não está nos documentos e que o juiz marca como não fiel. Ou o contrário: uma resposta fiel que formula o facto com outras palavras ("trinta dias" em vez de "30 dias") e falha a correspondência de texto. Lendo as respostas, nenhum dos dois métodos acerta sempre: a correspondência de texto só verifica um facto, e o juiz, sendo um modelo pequeno, também se engana, aceita afirmações não suportadas ou devolve JSON inválido. Um modelo pequeno como juiz serve para triagem, mas tem de ser validado com anotação humana e, de preferência, substituído por um modelo maior.

**Resposta Q11:** Na pergunta sem resposta (o telemóvel do reitor), o sistema deve abster-se com a frase pedida: os documentos recuperados não têm essa informação e o prompt dá instruções explícitas para dizer "Não sei com base nos documentos disponíveis.". Quanto às citações, a coluna `invalid_cites` mostra se alguma resposta citou um documento que não foi recuperado. Com o modelo restrito aos documentos que lhe são dados, isso deve ser raro ou nulo.

## Part 7 · Results & report (10 min)

Run the cell below and answer in the markdown cell that follows.

In [ ]:
summary = pd.DataFrame(RESULTS).T
summary.loc["end-to-end", "answer_acc"] = round(ev.correct.mean(), 3)
summary.loc["end-to-end", "faithful"] = round((ev[ev.faithful.notna()].faithful == True).mean(), 3)
print("Backend:", BACKEND, HF_MODEL if BACKEND == "hf" else ANTHROPIC_MODEL)
display(summary)
summary[["Recall@1", "Recall@5", "MRR"]].dropna().plot.bar(rot=20, title="Retrieval quality by configuration")
plt.ylim(0, 1.05); plt.show()

### Report (write here)

1. **Best configuration** and by how much it beats dense-only on Recall@1 and MRR: A melhor configuração é a híbrida (BM25 + densa com RRF) seguida de reranking com o cross-encoder. Supera a pesquisa só densa sobretudo no Recall@1 e no MRR (ver tabela acima), porque o BM25 recupera os códigos e termos exatos que os embeddings falham e o reranking põe o documento certo em primeiro lugar. Com 16 perguntas, cada pergunta vale 0,0625, por isso só diferenças de várias perguntas são significativas.
2. **One failure you could not fix** today, and the technique from the lecture you would try next (query rewriting, contextual retrieval, GraphRAG, agentic loop…): As perguntas coloquiais que não partilham vocabulário com o regulamento, como "subir a nota de uma cadeira" (o artigo diz "melhoria de classificação" numa "unidade curricular"). Tentaria **query rewriting / multi-query**, reescrevendo a pergunta para a linguagem formal do regulamento antes de pesquisar, ou **contextual retrieval**, acrescentando a cada excerto um resumo e sinónimos ("subir nota", "cadeira") antes de o indexar.
3. **Context engineering:** one change from Part 1 you would apply to a real chatbot, and why: Selecionar dinamicamente as ferramentas e os documentos em cada pedido, em vez de enviar tudo. Na Parte 1 as ferramentas ocupavam quase metade da janela e a pergunta menos de 1%. Reduzir o contexto ao essencial baixa o custo e a latência e diminui a distração e a confusão do modelo. Também ordenaria o prompt com as partes estáveis (sistema, ferramentas) no início, para aproveitar a cache.

## ★ Bonus (at home or for fast teams)

**B1 · Query rewriting for follow-ups.** `"E posso pagar tudo de uma vez?"` retrieves nothing useful on its own. Rewrite it into a standalone query using the conversation history, then retrieve.

In [ ]:
def rewrite_query(history: list, follow_up: str) -> str:
    prompt = ("Histórico da conversa:\n" + "\n".join(history) +
              f"\n\nPergunta de seguimento: {follow_up}\n\n"
              "Reescreve a pergunta de seguimento como uma pergunta autónoma e completa, em português europeu, "
              "que se perceba sem o histórico. Não respondas à pergunta. Devolve apenas a pergunta reescrita.")
    return llm(prompt, max_tokens=60).strip().strip('"').strip()

hist = ["Utilizador: Quanto custa a propina anual em 2026/2027?",
        "Assistente: A propina anual é de 710 euros [propinas-2026]."]
fu = "E posso pagar tudo de uma vez?"
rq = rewrite_query(hist, fu)
print("raw      :", hybrid_rerank(fu, k=3))
print("rewritten:", rq, "->", hybrid_rerank(rq, k=3))

**B2 · HyDE.** Generate a hypothetical answer, embed it and search with it. Evaluate as `"hyde"`. Does it help on the paraphrase questions (compare `MISSES["hyde"]` with `MISSES["dense"]`)? What does it cost?

In the original method the hypothetical answer is embedded **as a document**, so with E5 it takes the `passage: ` prefix.

In [ ]:
def hyde_search(query: str, k: int = 10) -> list:
    hypo = llm(f"Escreve um parágrafo curto, no estilo de um regulamento universitário em português europeu, "
               f"que responda à pergunta: {query}", max_tokens=80)
    h = emb_model.encode(["passage: " + hypo], normalize_embeddings=True)[0]
    scores = P_EMB @ h
    return [CHUNKS[i]["id"] for i in np.argsort(-scores)[:k]]

print(evaluate(hyde_search, "hyde"))
print("dense misses:", MISSES["dense"])
print("hyde misses :", MISSES["hyde"])

**B3 · A minimal agentic loop.** If the judge says the answer is unfaithful, or the model abstains, rewrite the query and retrieve again (at most 2 attempts). Log every step. Where could this loop go wrong in production?

In [ ]:
def agentic_answer(question: str, max_steps: int = 2) -> dict:
    q, log = question, []
    for step in range(max_steps):
        a = rag_answer(q)
        verdict = judge(a)
        log.append({"step": step, "query": q, "answer": a["answer"], "faithful": verdict.get("faithful")})
        if verdict.get("faithful") and not a["abstained"]:
            break
        prompt = (f"Pergunta original: {question}\nÚltima pesquisa: {q}\n\n"
                  "A pesquisa não encontrou uma resposta fiável. Escreve uma pergunta de pesquisa alternativa, "
                  "em português europeu, com outras palavras mas com o mesmo significado da pergunta original. "
                  "Devolve apenas a pergunta.")
        q = llm(prompt, max_tokens=60).strip().strip('"\'').strip()
    return {"final": log[-1]["answer"], "log": log}

res = agentic_answer("Quantas vezes posso tentar subir a nota de uma cadeira?")
display(pd.DataFrame(res["log"])[["step", "query", "faithful"]])
for s in res["log"]:
    print(f"\n[step {s['step']}] {s['query']}\n    {s['answer']}")

**B2 · HyDE:** A HyDE pode ajudar nas perguntas de paráfrase, porque a resposta hipotética é escrita no estilo do regulamento ("melhoria de classificação", "unidade curricular") e fica mais perto dos documentos certos no espaço vetorial (comparar `MISSES["hyde"]` com `MISSES["dense"]`). O custo é uma chamada extra ao LLM por pergunta, ou seja, muito mais latência, e o risco de a resposta inventada puxar a pesquisa para o sítio errado quando o modelo alucina.

**B3 · Onde o ciclo agêntico pode correr mal em produção:** o juiz pode enganar-se e pedir novas pesquisas para respostas que estavam certas, ou aceitar respostas erradas; as reformulações podem afastar-se da pergunta original (drift); cada passo multiplica a latência e o custo; e sem um limite de passos e registos claros, o sistema pode entrar em ciclos difíceis de depurar.